In [ ]:
import cv2
import numpy as np

def detect_coins_and_create_mask(image_path):
    """
    Loads an image, detects coins using the Hough Circle Transform,
    and returns a binary mask of the detected coins.
    """

    # --- 1. Load and Pre-process the Image ---

    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image from {image_path}")
        return

    # Create a copy to draw on later (for visualization)
    output_image = image.copy()

    # Create a blank (black) image for the binary mask
    # It has the same height and width, but is a single channel (grayscale)
    mask = np.zeros(image.shape[:2], dtype="uint8")

    # Convert the image to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian blur to reduce noise and improve detection
    # A (9, 9) kernel is used here; this may need tuning.
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)

    # --- 2. Detect Circles using Hough Transform ---

    # cv2.HoughCircles parameters:
    # 1. image: The input grayscale image.
    # 2. method: cv2.HOUGH_GRADIENT - the only method available.
    # 3. dp: Inverse ratio of accumulator resolution. 1.2 works well.
    # 4. minDist: Minimum distance between detected circle centers.
    # 5. param1: Upper threshold for the Canny edge detector.
    # 6. param2: Threshold for circle center detection. (Smaller = more circles)
    # 7. minRadius: Smallest coin radius to detect.
    # 8. maxRadius: Largest coin radius to detect.

    # These parameters are tuned for your specific image.
    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=70,
        param1=70,
        param2=60,
        minRadius=30,
        maxRadius=120
    )

    # --- 3. Create the Binary Mask and Visualization ---

    if circles is not None:
        # Convert circle parameters (x, y, radius) to integers
        circles = np.round(circles[0, :]).astype("int")

        print(f"Detected {len(circles)} coins.")

        # Loop over the detected circles
        for (x, y, r) in circles:
            # --- For the Binary Mask ---
            # Draw a FILLED circle (white) on the black mask image.
            # Thickness = -1 means the circle is filled.
            cv2.circle(mask, (x, y), r, 255, -1)

            # --- For Visualization ---
            # Draw the (green) outline of the circle on the output image
            cv2.circle(output_image, (x, y), r, (0, 255, 0), 4)
            # Draw a (red) small circle for the center
            cv2.circle(output_image, (x, y), 2, (0, 0, 255), 3)

        # Save the results
        cv2.imwrite("coin_binary_mask.png", mask)
        cv2.imwrite("coin_detection_visualization.png", output_image)

        print("\nSuccessfully saved:")
        print("1. 'coin_binary_mask.png' (Your requested binary mask)")
        print("2. 'coin_detection_visualization.png' (A visualization of the results)")

    else:
        print("No circles were detected.")

# --- 4. Run the Code ---
if __name__ == "__main__":
    # Use the filename of the image you uploaded
    image_file = "/content/coins_d76faad9-e401-4198-a666-59ecd02f26be.jpg"
    detect_coins_and_create_mask(image_file)

Detected 8 coins.

Successfully saved:
1. 'coin_binary_mask.png' (Your requested binary mask)
2. 'coin_detection_visualization.png' (A visualization of the results)
